# Data Preparation

This notebook checks and prepares the anime dataset for the other notebooks.

In [2]:
import pandas as pd

df = pd.read_csv("../data/popular_anime.csv")

print("Rows and columns:", df.shape)
df.head()

Rows and columns: (28825, 18)


,id,name,genres,type,episodes,status,aired_from,aired_to,duration_per_ep,score,scored_by,rank,rating,studios,producers,image,trailer,synopsis
0,52991,Frieren: Beyond Journey's End,"Adventure, Drama, Fantasy",TV,28.0,Finished Airing,2023-09-29T00:00:00+00:00,2024-03-22T00:00:00+00:00,24 min per ep,9.30,676737.0,1.0,PG-13 - Teens 13 or older,Madhouse,"Aniplex, Dentsu, Shogakukan-Shueisha Productio...",https://cdn.myanimelist.net/images/anime/1015/...,https://www.youtube.com/watch?v=ZEkwCGJ3o7M,During their decade-long quest to defeat the D...
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy",TV,64.0,Finished Airing,2009-04-05T00:00:00+00:00,2010-07-04T00:00:00+00:00,24 min per ep,9.10,2223666.0,2.0,R - 17+ (violence & profanity),Bones,"Aniplex, Square Enix, Mainichi Broadcasting Sy...",https://cdn.myanimelist.net/images/anime/1208/...,https://www.youtube.com/watch?v=1ac3_YdSSy0,After a horrific alchemy experiment goes wrong...
2,9253,Steins;Gate,"Drama, Sci-Fi, Suspense",TV,24.0,Finished Airing,2011-04-06T00:00:00+00:00,2011-09-14T00:00:00+00:00,24 min per ep,9.07,1467570.0,3.0,PG-13 - Teens 13 or older,White Fox,"Frontier Works, Media Factory, Kadokawa Shoten...",https://cdn.myanimelist.net/images/anime/1935/...,https://www.youtube.com/watch?v=27OZc-ku6is,Eccentric scientist Rintarou Okabe has a never...
3,38524,Attack on Titan Season 3 Part 2,"Action, Drama, Suspense",TV,10.0,Finished Airing,2019-04-29T00:00:00+00:00,2019-07-01T00:00:00+00:00,23 min per ep,9.05,1700946.0,4.0,R - 17+ (violence & profanity),Wit Studio,"Production I.G, Dentsu, Mainichi Broadcasting ...",https://cdn.myanimelist.net/images/anime/1517/...,https://www.youtube.com/watch?v=hKHepjfj5Tw,Seeking to restore humanity's diminishing hope...
4,28977,Gintama Season 4,"Action, Comedy, Sci-Fi",TV,51.0,Finished Airing,2015-04-08T00:00:00+00:00,2016-03-30T00:00:00+00:00,24 min per ep,9.05,264260.0,5.0,PG-13 - Teens 13 or older,Bandai Namco Pictures,"TV Tokyo, Aniplex, Dentsu",https://cdn.myanimelist.net/images/anime/3/720...,NaN,"Gintoki, Shinpachi, and Kagura return as the f..."


## Check the data

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28825 entries, 0 to 28824
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               28825 non-null  int64  
 1   name             28825 non-null  str    
 2   genres           22756 non-null  str    
 3   type             28738 non-null  str    
 4   episodes         28046 non-null  float64
 5   status           28825 non-null  str    
 6   aired_from       27870 non-null  str    
 7   aired_to         10930 non-null  str    
 8   duration_per_ep  28825 non-null  str    
 9   score            18599 non-null  float64
 10  scored_by        18599 non-null  float64
 11  rank             21962 non-null  float64
 12  rating           28101 non-null  str    
 13  studios          16936 non-null  str    
 14  producers        13452 non-null  str    
 15  image            28825 non-null  str    
 16  trailer          5474 non-null   str    
 17  synopsis         23457 

In [4]:
# Missing values in each column
df.isna().sum().sort_values(ascending=False)

trailer            23351
aired_to           17895
producers          15373
studios            11889
scored_by          10226
score              10226
rank                6863
genres              6069
synopsis            5368
aired_from           955
episodes             779
rating               724
type                  87
id                     0
name                   0
status                 0
duration_per_ep        0
image                  0
dtype: int64

In [5]:
# Check duplicates
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["id"].duplicated().sum())

Duplicate rows: 218
Duplicate IDs: 218


## Basic cleaning

There are repeated IDs, so I will keep the first record for each ID. I will also remove extra spaces from the text columns.

I am not removing all rows with missing values because different analyses will use different columns.

In [6]:
df = df.drop_duplicates(subset="id", keep="first").copy()

text_columns = [
    "name", "genres", "type", "status", "duration_per_ep",
    "rating", "studios", "producers"
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)

print("Shape after cleaning:", df.shape)

Shape after cleaning: (28607, 18)


## Check numeric columns

Convert the main numeric columns in case any values have been stored as text.

In [7]:
numeric_columns = ["id", "episodes", "score", "scored_by", "rank"]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_columns].describe()

,id,episodes,score,scored_by,rank
count,28607.000000,27924.000000,18541.000000,1.854100e+04,21879.000000
mean,33931.913133,13.856503,6.396148,2.997605e+04,10522.296814
std,19557.455957,46.936902,0.889713,1.214772e+05,6382.819523
min,1.000000,1.000000,1.890000,1.010000e+02,0.000000
25%,15705.000000,1.000000,5.780000,3.400000e+02,4992.500000
50%,37945.000000,2.000000,6.370000,1.552000e+03,10463.000000
75%,50584.500000,13.000000,7.030000,1.013200e+04,16016.500000
max,61871.000000,3057.000000,9.300000,2.943048e+06,21729.000000


## Dates

Convert the airing dates and create a year column. This will be used later to look at anime releases over time.

In [8]:
df["aired_from"] = pd.to_datetime(df["aired_from"], errors="coerce", utc=True)
df["aired_to"] = pd.to_datetime(df["aired_to"], errors="coerce", utc=True)

df["release_year"] = df["aired_from"].dt.year

df[["name", "aired_from", "aired_to", "release_year"]].head()

,name,aired_from,aired_to,release_year
0,Frieren: Beyond Journey's End,2023-09-29 00:00:00+00:00,2024-03-22 00:00:00+00:00,2023.0
1,Fullmetal Alchemist: Brotherhood,2009-04-05 00:00:00+00:00,2010-07-04 00:00:00+00:00,2009.0
2,Steins;Gate,2011-04-06 00:00:00+00:00,2011-09-14 00:00:00+00:00,2011.0
3,Attack on Titan Season 3 Part 2,2019-04-29 00:00:00+00:00,2019-07-01 00:00:00+00:00,2019.0
4,Gintama Season 4,2015-04-08 00:00:00+00:00,2016-03-30 00:00:00+00:00,2015.0


## Check and save the prepared data

In [9]:
print("Final shape:", df.shape)
df.isna().sum().sort_values(ascending=False).head(10)

Final shape: (28607, 19)


trailer       23231
aired_to      17714
producers     15266
studios       11816
scored_by     10066
score         10066
rank           6728
genres         6050
synopsis       5332
aired_from      892
dtype: int64

In [10]:
# Save the cleaned data in the data folder
df.to_csv("../data/cleaned_popular_anime.csv", index=False)

print("Saved cleaned_popular_anime.csv")

Saved cleaned_popular_anime.csv


The prepared dataset is saved in the `data` folder. The original CSV has not been changed.